# NiyamTrace-X Q1 Experiment 1 — Semantic Anchor Lock Stress Benchmark

**Goal:** add a large multilingual controlled-corruption experiment for the manuscript's
surface-to-contract safety claim.

It targets entity substitution, temporal drift, amount/currency drift, negation deletion,
quantifier drift, and hallucinated critical slots across English, Romanized Hindi,
Romanized Telugu, and Telugu script.

This is CPU-only and API-free. It is a **new controlled systems experiment**, not a
reconstruction of the missing frozen 2,000-case model run.

**Outputs:** case-level CSV, per-language/per-relation metrics, confidence intervals,
LaTeX table, an external/human-validation CSV template, and a ZIP bundle.

In [ ]:
# Reproducible setup: pin the exact public repository commit audited in the manuscript.
REPO_URL = "https://github.com/bnssaanirudh/NiyamTrace-X.git"
PINNED_COMMIT = "c14661dbd11c42ebd1019b6a1a5c49b8643da137"

!rm -rf /content/NiyamTrace-X
!git clone -q $REPO_URL /content/NiyamTrace-X
%cd /content/NiyamTrace-X
!git checkout -q $PINNED_COMMIT

!pip -q install -e /content/NiyamTrace-X/niyamtrace
!pip -q install pandas numpy scipy scikit-learn matplotlib tqdm statsmodels nbformat

from pathlib import Path
import os, json, math, random, hashlib, statistics, itertools, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/content/NiyamTrace-X")
NIYAM = ROOT / "niyamtrace"
RESULTS = Path("/content/niyamtrace_q1_results")
RESULTS.mkdir(exist_ok=True)
print("Pinned commit:", PINNED_COMMIT)
print("Results:", RESULTS)

In [ ]:
SEED = 20260911
BASE_PER_LANGUAGE = 600
rng = random.Random(SEED)
np.random.seed(SEED)

LANGUAGES = ["eng_Latn", "hin_Latn", "tel_Latn", "tel_Telu"]
RELATIONS = [
    "CLEAN", "ENTITY_SHIFT", "TEMPORAL_SHIFT", "VALUE_SHIFT",
    "NEGATION_DROP", "QUANTIFIER_DROP", "HALLUCINATED_CRITICAL_SLOT"
]
MONTHS_EN = {
    1:"january",2:"february",3:"march",4:"april",5:"may",6:"june",
    7:"july",8:"august",9:"september",10:"october",11:"november",12:"december"
}

def render(lang, vendor, month, year, amount, negated, quantifier):
    q = {
        "eng_Latn": "all" if quantifier == "ALL" else "only",
        "hin_Latn": "sabhi" if quantifier == "ALL" else "sirf",
        "tel_Latn": "anni" if quantifier == "ALL" else "maatrame",
        "tel_Telu": "అన్ని" if quantifier == "ALL" else "మాత్రమే",
    }[lang]
    neg = {
        "eng_Latn": "do not " if negated else "",
        "hin_Latn": "mat " if negated else "",
        "tel_Latn": "cheyyaku, " if negated else "",
        "tel_Telu": "చేయవద్దు, " if negated else "",
    }[lang]
    if lang == "eng_Latn":
        return f"{neg}archive {q} invoices for vendor {vendor} for {MONTHS_EN[month]} {year} with amount INR {amount}."
    if lang == "hin_Latn":
        return f"Vendor {vendor} ke {month}/{year} ke INR {amount} wale {q} invoices {neg}archive karo."
    if lang == "tel_Latn":
        return f"Vendor {vendor} yoka {month}/{year} INR {amount} {q} invoices {neg}archive cheyyandi."
    return f"Vendor {vendor} కి {month}/{year} INR {amount} {q} invoices {neg}archive చేయండి."

def make_contract(vendor, month, year, amount, negated, quantifier):
    return {
        "VENDOR_ID": int(vendor), "MONTH": int(month), "YEAR": int(year),
        "AMOUNT": int(amount), "CURRENCY": "INR",
        "NEGATED": bool(negated), "QUANTIFIER": quantifier
    }

def corrupt(c, relation):
    out = dict(c)
    if relation == "ENTITY_SHIFT":
        out["VENDOR_ID"] += rng.choice([1, 7, 101])
    elif relation == "TEMPORAL_SHIFT":
        out["MONTH"] = 1 + (out["MONTH"] % 12)
    elif relation == "VALUE_SHIFT":
        out["AMOUNT"] += rng.choice([500, 1000, 5000])
    elif relation == "NEGATION_DROP":
        out["NEGATED"] = False
    elif relation == "QUANTIFIER_DROP":
        out["QUANTIFIER"] = "MATCHING"
    elif relation == "HALLUCINATED_CRITICAL_SLOT":
        out["RECIPIENT_ID"] = f"R{rng.randint(100,999)}"
    return out

rows = []
for lang in LANGUAGES:
    for i in range(BASE_PER_LANGUAGE):
        vendor = rng.randint(1000, 9999)
        month = rng.randint(1, 12)
        year = rng.choice([2024, 2025, 2026])
        amount = rng.choice([5000, 7500, 10000, 25000, 50000, 100000])
        negated = True  # every base explicitly carries negation so NEGATION_DROP is a real corruption
        quantifier = "ALL" if (i % 3 == 0) else "ONLY"
        raw = render(lang, vendor, month, year, amount, negated, quantifier)
        gold = make_contract(vendor, month, year, amount, negated, quantifier)
        for relation in RELATIONS:
            candidate = dict(gold) if relation == "CLEAN" else corrupt(gold, relation)
            rows.append({
                "case_id": f"{lang}-{i:04d}-{relation}",
                "group_id": f"{lang}-{i:04d}",
                "language": lang,
                "relation": relation,
                "raw_text": raw,
                "gold_contract": json.dumps(gold, sort_keys=True, ensure_ascii=False),
                "candidate_contract": json.dumps(candidate, sort_keys=True, ensure_ascii=False),
                "expected_block": relation != "CLEAN",
            })

cases = pd.DataFrame(rows)
print("Cases:", len(cases))
display(cases.head(3))

In [ ]:
NEG_PATTERNS = {
    "eng_Latn": [r"\bdo not\b", r"\bdon't\b", r"\bnever\b"],
    "hin_Latn": [r"\bmat\b", r"\bnahi\b", r"\bnahin\b"],
    "tel_Latn": [r"\bcheyyaku\b", r"\bvaddu\b", r"\bkaadu\b"],
    "tel_Telu": [r"చేయవద్దు", r"వద్దు", r"కాదు"],
}
ALL_PATTERNS = {
    "eng_Latn": [r"\ball\b", r"\bevery\b"],
    "hin_Latn": [r"\bsabhi\b", r"\bsaare\b"],
    "tel_Latn": [r"\banni\b"],
    "tel_Telu": [r"అన్ని"],
}
ONLY_PATTERNS = {
    "eng_Latn": [r"\bonly\b"],
    "hin_Latn": [r"\bsirf\b"],
    "tel_Latn": [r"\bmaatrame\b"],
    "tel_Telu": [r"మాత్రమే"],
}

def any_match(patterns, text):
    import re
    return any(re.search(p, text, flags=re.I) for p in patterns)

def extract_surface_anchors(text, lang):
    import re
    anchors = {}

    m = re.search(r"vendor\s+(\d{3,10})", text, flags=re.I)
    if m:
        anchors["VENDOR_ID"] = int(m.group(1))

    my = re.search(r"\b(1[0-2]|0?[1-9])\s*/\s*(20\d{2})\b", text)
    if my:
        anchors["MONTH"], anchors["YEAR"] = int(my.group(1)), int(my.group(2))
    else:
        low = text.lower()
        for month, name in MONTHS_EN.items():
            if re.search(rf"\b{name}\b", low):
                anchors["MONTH"] = month
                y = re.search(r"\b(20\d{2})\b", text)
                if y:
                    anchors["YEAR"] = int(y.group(1))
                break

    amt = re.search(r"\b(?:INR|₹)\s*([0-9][0-9,]*)\b", text, flags=re.I)
    if amt:
        anchors["AMOUNT"] = int(amt.group(1).replace(",", ""))
        anchors["CURRENCY"] = "INR"

    anchors["NEGATED"] = any_match(NEG_PATTERNS[lang], text)
    if any_match(ALL_PATTERNS[lang], text):
        anchors["QUANTIFIER"] = "ALL"
    elif any_match(ONLY_PATTERNS[lang], text):
        anchors["QUANTIFIER"] = "ONLY"

    return anchors

def semantic_anchor_lock(text, lang, candidate):
    surface = extract_surface_anchors(text, lang)
    violations = []

    for key in ("VENDOR_ID", "MONTH", "YEAR", "AMOUNT"):
        if key in surface and candidate.get(key) != surface[key]:
            violations.append(f"{key}_DRIFT")

    if surface.get("NEGATED") is True and candidate.get("NEGATED") is not True:
        violations.append("NEGATION_DROP")

    if "QUANTIFIER" in surface and candidate.get("QUANTIFIER") != surface["QUANTIFIER"]:
        violations.append("QUANTIFIER_DRIFT")

    if surface.get("CURRENCY") == "INR" and candidate.get("CURRENCY") != "INR":
        violations.append("CURRENCY_DRIFT")

    # A security-critical recipient that appears only in the generated contract and not
    # in the user's surface instruction is treated as a hallucinated critical slot.
    if candidate.get("RECIPIENT_ID") is not None:
        violations.append("HALLUCINATED_RECIPIENT")

    return {"block": bool(violations), "violations": violations, "surface_anchors": surface}

# Sanity checks covering safe and corrupted cases.
safe = cases[cases.relation=="CLEAN"].iloc[0]
bad = cases[cases.relation=="ENTITY_SHIFT"].iloc[0]
assert not semantic_anchor_lock(safe.raw_text, safe.language, json.loads(safe.candidate_contract))["block"]
assert semantic_anchor_lock(bad.raw_text, bad.language, json.loads(bad.candidate_contract))["block"]
print("Anchor Lock sanity checks passed.")

In [ ]:
records = []
for r in cases.itertuples(index=False):
    candidate = json.loads(r.candidate_contract)
    out = semantic_anchor_lock(r.raw_text, r.language, candidate)
    records.append({
        "case_id": r.case_id, "group_id": r.group_id, "language": r.language,
        "relation": r.relation, "expected_block": bool(r.expected_block),
        "predicted_block": bool(out["block"]), "violations": "|".join(out["violations"]),
    })

res = pd.DataFrame(records)
res["correct"] = res.expected_block == res.predicted_block

tp = int((res.expected_block & res.predicted_block).sum())
tn = int((~res.expected_block & ~res.predicted_block).sum())
fp = int((~res.expected_block & res.predicted_block).sum())
fn = int((res.expected_block & ~res.predicted_block).sum())

recall = tp/(tp+fn) if tp+fn else float("nan")
fpr = fp/(fp+tn) if fp+tn else float("nan")
accuracy = float(res.correct.mean())

overall = {
    "n": len(res), "accuracy": accuracy,
    "unsafe_corruption_recall": recall, "clean_false_positive_rate": fpr,
    "tp": tp, "tn": tn, "fp": fp, "fn": fn
}
by_lang = res.groupby("language").agg(
    n=("correct","size"), accuracy=("correct","mean"), blocks=("predicted_block","sum")
).reset_index()
by_rel = res.groupby("relation").agg(
    n=("correct","size"), accuracy=("correct","mean"), blocks=("predicted_block","sum")
).reset_index()

print(json.dumps(overall, indent=2))
display(by_lang)
display(by_rel)

In [ ]:
from statsmodels.stats.proportion import proportion_confint

def wilson(k, n, alpha=0.05):
    lo, hi = proportion_confint(k, n, alpha=alpha, method="wilson")
    return float(lo), float(hi)

unsafe = res[res.expected_block]
clean = res[~res.expected_block]
rec_lo, rec_hi = wilson(int(unsafe.predicted_block.sum()), len(unsafe))
spec_lo, spec_hi = wilson(int((~clean.predicted_block).sum()), len(clean))

summary = pd.DataFrame([{
    "N":len(res), "Accuracy":accuracy, "UnsafeRecall":recall,
    "UnsafeRecall_CI_L":rec_lo, "UnsafeRecall_CI_H":rec_hi,
    "CleanFPR":fpr, "Specificity_CI_L":spec_lo, "Specificity_CI_H":spec_hi
}])
display(summary)

fig, ax = plt.subplots(figsize=(8,4.5))
ax.bar(by_rel["relation"], by_rel["accuracy"])
ax.set_ylim(0,1.02)
ax.set_ylabel("Detection accuracy")
ax.set_title("Semantic Anchor Lock stress accuracy by corruption type")
ax.tick_params(axis="x", rotation=40)
fig.tight_layout()
fig.savefig(RESULTS/"anchor_accuracy_by_relation.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
cases.to_csv(RESULTS/"anchor_stress_cases.csv", index=False)
res.to_csv(RESULTS/"anchor_case_results.csv", index=False)
by_lang.to_csv(RESULTS/"anchor_by_language.csv", index=False)
by_rel.to_csv(RESULTS/"anchor_by_relation.csv", index=False)
summary.to_csv(RESULTS/"anchor_summary.csv", index=False)
(RESULTS/"anchor_overall_metrics.json").write_text(json.dumps(overall, indent=2))
(RESULTS/"anchor_table.tex").write_text(summary.to_latex(index=False, float_format=lambda x:f"{x:.5f}"))

external_template = pd.DataFrame(columns=[
    "case_id","group_id","language","relation","raw_text",
    "candidate_contract","expected_block","annotator_1","annotator_2","adjudicated"
])
external_template.to_csv(RESULTS/"anchor_external_template.csv", index=False)

gates = {
    "unsafe_recall_ge_0_99": recall >= 0.99,
    "clean_fpr_le_0_01": fpr <= 0.01,
    "every_relation_accuracy_ge_0_98": bool((by_rel.accuracy >= 0.98).all()),
}
(RESULTS/"anchor_acceptance_gates.json").write_text(json.dumps(gates, indent=2))
print(json.dumps(gates, indent=2))

In [ ]:
# Optional genuinely external/human-authored CSV evaluation.
# Put the completed template anywhere under /content with "external" and "anchor" in its filename.
ext_files = list(Path("/content").glob("*external*anchor*.csv"))
if not ext_files:
    print("No external anchor CSV found. Controlled stress results remain INTERNAL.")
else:
    ext = pd.read_csv(ext_files[0])
    out_rows = []
    for r in ext.itertuples(index=False):
        cand = json.loads(r.candidate_contract)
        out = semantic_anchor_lock(r.raw_text, r.language, cand)
        d = r._asdict()
        d["predicted_block"] = out["block"]
        out_rows.append(d)
    ext_res = pd.DataFrame(out_rows)
    ext_res["correct"] = ext_res.expected_block.astype(bool) == ext_res.predicted_block.astype(bool)
    display(ext_res.groupby(["language","relation"]).correct.agg(["size","mean"]))
    ext_res.to_csv(RESULTS/"anchor_EXTERNAL_results.csv", index=False)

In [ ]:
import zipfile
zip_path = Path("/content/NTX_Q1_01_Anchor_Stress_RESULTS.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in RESULTS.glob("anchor_*"):
        z.write(p, arcname=p.name)
print(zip_path)

In [ ]:
# DOWNLOAD RESULTS ZIP
from pathlib import Path
from google.colab import files

download_zip = Path("/content/NTX_Q1_01_Anchor_Stress_RESULTS.zip")

if not download_zip.exists():
    raise FileNotFoundError(
        f"{download_zip} was not found. Run the result-export/ZIP cell above first."
    )

print(f"Downloading: {download_zip.name}")
files.download(str(download_zip))
